# Časť 2, Téma 2: Obídenie hesla pomocou Voltage Glitchingu

---
POZNÁMKA: Toto praktické cvičenie odkazuje na niektoré (komerčné) školiace materiály na webe [ChipWhisperer.io](https://www.ChipWhisperer.io). Cvičenie môžete voľne spustiť a používať v súlade s licenciou open-source (vrátane jeho použitia vo vašich vlastných kurzoch, ak ich budete distribuovať podobným spôsobom), musíte však zachovať informáciu o tomto pôvodnom zdroji. Ak si chcete vyskúšať plnohodnotný zážitok, zvážte účasť na našom školení.

---

**ZHRNUTIE:** *Už sme videli, ako sa dá vkladanie chýb do napájania (voltage glitching) použiť na narušenie výpočtov, rovnako ako pri hodinovom glitchovaní (clock glitching). Poďme pokračovať ďalej a pozrime sa, či sa dá táto metóda využiť aj na prelomenie kontroly hesla.*

**VÝSLEDKY VZDELÁVANIA:**

* Aplikovanie predchádzajúcich nastavení glitchovania na nový firmvér
* Kontrola úspešnosti a zlyhania pri vykonávaní glitchu

## Firmvér

Keďže sme toto cvičenie už raz preberali, tentokrát to bude prevažne na vás!

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEXMEGA'
SS_VER = 'SS_VER_2_1'

In [ ]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../firmware/mcu/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

In [ ]:
fw_path = "../../firmware/mcu/simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER=="SS_VER_2_1":
    target.reset_comms()

In [ ]:
def reboot_flush():
    reset_target(scope)
    target.flush()
if PLATFORM == "CWLITEXMEGA":
    scope.clock.clkgen_freq = 32E6
    if SS_VER=='SS_VER_2_1':
        target.baud = 230400*32/7.37
    else:
        target.baud = 38400*32/7.37
elif (PLATFORM == "CWLITEARM") or ("F3" in PLATFORM):
    scope.clock.clkgen_freq = 24E6
    if SS_VER=='SS_VER_2_1':
        target.baud = 230400*24/7.37
    else:
        target.baud = 38400*24/7.37
    

In [ ]:
#Do glitch loop
reboot_flush()
pw = bytearray([0x74, 0x6F, 0x75, 0x63, 0x68])
target.simpleserial_write('p', pw)

val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)

Podobne ako pri hodinovom glitchovaní, objekt `scope` dokáže automaticky pripraviť niektoré typické konfigurácie pre napäťový glitch:

In [ ]:
if scope._is_husky:
    scope.vglitch_setup('hp', default_setup=False) # HP alone works best for Husky
else:
    scope.vglitch_setup('both', default_setup=False) # use both transistors

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

In [ ]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None})

In [ ]:
import re
import struct
gc.set_range("ext_offset", 0, 150)
g_step = 0.4
gc.set_global_step(g_step)

#disable logging
cw.set_all_log_levels(cw.logging.CRITICAL)

gc.set_range("width", ???, ???)
gc.set_range("offset", ???, ???)
gc.set_global_step(???)

gc.set_step("ext_offset", 1) # check each clock cycle

scope.adc.timeout = 0.1

reboot_flush()

successes = 0

for glitch_settings in gc.glitch_values():
    scope.glitch.offset = glitch_settings[1]
    scope.glitch.width = glitch_settings[0]
    scope.glitch.ext_offset = glitch_settings[2]
    if scope.adc.state:
        # can detect crash here (fast) before timing out (slow)
        #print("Trigger still high!")
        gc.add("reset")
        reboot_flush()
 
    scope.arm()
    target.simpleserial_write('p', bytearray([0]*5))
    ret = scope.capture()
    scope.io.vglitch_reset()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")
 
        #Device is slow to boot?
        reboot_flush()
    else:
        val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50)#For loop check
        if ???:
            gc.add("reset")
        else:
            if ???: #for loop check
                successes +=1 
                gc.add("success")
                print(val)
                print(val['payload'])
                print(scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                print("🐙", end="")
            else:
                gc.add("normal")
                    
#reenable logging
cw.set_all_log_levels(cw.logging.WARNING)

Poďme sa pozrieť, na ktoré miesta sme museli zacieliť, aby náš glitch zafungoval:

In [ ]:
gc.calc(["width", "offset"], "success_rate")

In [ ]:
scope.dis()
target.dis()